In [ ]:
"""
Standardized Precipitation Index (SPI) Calculator

Core functions to compute SPI using Gamma distribution fitting on precipitation
accumulations over a specified timescale. Follows standard WMO methodology with
mixed distribution handling for zero values.
"""

import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import gamma, norm
from scipy.ndimage import convolve1d
from typing import Union


def compute_spi_timeseries(
    precipitation: np.ndarray,
    timescale: int = 360,
    skip_initial: int = 365
) -> np.ndarray:
    """
    Compute SPI for a single precipitation timeseries using Gamma distribution.
    
    Implements standard SPI methodology:
      1. Accumulate precipitation over specified timescale (rolling window)
      2. Fit Gamma distribution to non-zero accumulations
      3. Transform to normal distribution using mixed probability approach
    
    Parameters
    ----------
    precipitation : np.ndarray
        1D array of daily precipitation values (mm/day)
    timescale : int
        Accumulation period in days (e.g., 30, 90, 360)
    skip_initial : int
        Number of initial values to discard (for accumulation warm-up)
    
    Returns
    -------
    np.ndarray
        SPI values aligned to input timeseries (after skip_initial offset).
        Length = len(precipitation) - skip_initial
    """
    # Replace negative values with zero (physically invalid)
    precip = np.maximum(precipitation.astype(float), 0.0)
    
    # Compute rolling accumulations
    weights = np.ones(timescale) / timescale
    accumulations = convolve1d(precip, weights=weights, mode='constant', cval=0.0)
    
    # Apply warm-up skip
    accumulations = accumulations[skip_initial:]
    
    # Handle all-zero case
    if np.all(accumulations == 0) or np.all(np.isnan(accumulations)):
        return np.full(len(accumulations), np.nan)
    
    # Mixed distribution approach (WMO standard)
    zero_ratio = np.sum(accumulations == 0) / len(accumulations)
    nonzero_vals = accumulations[accumulations > 0]
    
    if len(nonzero_vals) < 2:
        return np.full(len(accumulations), np.nan)
    
    # Fit Gamma distribution to non-zero values (fix location at zero)
    shape, _, scale = gamma.fit(nonzero_vals, floc=0)
    
    # Compute cumulative probabilities with zero-inflation adjustment
    cdf_vals = gamma.cdf(accumulations, shape, loc=0, scale=scale)
    mixed_cdf = zero_ratio + (1 - zero_ratio) * cdf_vals
    
    # Transform to standard normal (SPI)
    spi = norm.ppf(mixed_cdf)
    spi[np.isnan(accumulations)] = np.nan
    return spi


def calculate_spi_dataset(
    precipitation_ds: xr.Dataset,
    timescale: int = 360,
    precip_var: str = 'tp',
    time_dim: str = 'date',
    skip_initial: int = 365
) -> xr.Dataset:
    """
    Compute SPI across all grid cells in a precipitation dataset.
    
    Parameters
    ----------
    precipitation_ds : xr.Dataset
        Dataset containing precipitation variable with dimensions (time, latitude, longitude)
    timescale : int
        SPI accumulation period in days (default: 360)
    precip_var : str
        Name of precipitation variable in dataset (default: 'tp')
    time_dim : str
        Name of time dimension (default: 'date')
    skip_initial : int
        Days to skip at start for accumulation warm-up (default: 365)
    
    Returns
    -------
    xr.Dataset
        Dataset containing SPI values with same spatial dimensions and reduced time dimension
    """
    precip = precipitation_ds[precip_var]
    time_coords = pd.to_datetime(precip[time_dim].values)
    
    # Initialize output array (time dimension reduced by skip_initial)
    n_times = len(time_coords) - skip_initial
    spi_array = np.full((n_times, len(precip.latitude), len(precip.longitude)), np.nan)
    
    # Process each grid cell
    for i, lat in enumerate(precip.latitude.values):
        for j, lon in enumerate(precip.longitude.values):
            ts = precip.sel(latitude=lat, longitude=lon).values
            spi_array[:, i, j] = compute_spi_timeseries(
                ts, 
                timescale=timescale, 
                skip_initial=skip_initial
            )
    
    # Construct output dataset
    spi_ds = xr.Dataset(
        {'spi': ((time_dim, 'latitude', 'longitude'), spi_array)},
        coords={
            time_dim: time_coords[skip_initial:],
            'latitude': precip.latitude,
            'longitude': precip.longitude
        }
    )
    
    # Add metadata
    spi_ds['spi'].attrs.update({
        'long_name': 'Standardized Precipitation Index',
        'units': 'dimensionless',
        'timescale_days': timescale,
        'method': 'Gamma distribution with zero-inflation adjustment',
        'skip_initial_days': skip_initial
    })
    
    return spi_ds